In [14]:
import numpy as np
import keras
from keras import layers, ops

In [15]:
VOCAB = {"A": 1, "C": 2, "G": 3, "U": 4, "EMPTY": 0}

In [16]:
def encode(seq):
    """Convert an RNA sequence into a list of integer token ids.
    Parameters:
    -----------
        seq (str): RNA sequence made of the letters A, C, G and U.
        
    Returns:
    --------
        list[int]: One token id per nucleotide, in the same order as `seq`.
    """
    return [VOCAB[nuc] for nuc in seq]

In [17]:
def fill_empty(list_of_sequences):
    number_of_sequence = len(list_of_sequences)
    seq_length_max = 0
    for sequence in list_of_sequences:
        length = len(sequence)
        if length > seq_length_max:
            seq_length_max = length
            
    tokens = np.full((number_of_sequence, seq_length_max), VOCAB["EMPTY"])
    
    for line, seq in enumerate(list_of_sequences):
        tokens[line,0 :len(seq)] = seq
    mask = tokens != VOCAB["EMPTY"]
    return tokens, mask

In [18]:
class NucleotideEmbedding(layers.Layer):
    def __init__(self, dim=192, **kwargs):
        super().__init__(**kwargs)
        self.emb = layers.Embedding(len(VOCAB), dim, mask_zero=True)

    def call(self, tokens):
        return self.emb(tokens)

In [30]:
import numpy as np

def create_bpp_matrix(bpp_file, size):
    grille = np.zeros((size, size))

    with open(bpp_file) as f:
        for ligne in f:
            if not ligne.strip():
                continue

            i, j, proba = ligne.split()
            i, j = int(i) - 1, int(j) - 1

            grille[i, j] = float(proba)
            grille[j, i] = float(proba)

    return grille


In [ ]:
def build_mini_model(dim, n_outputs):
    tokens = keras.Input(shape=(None,), dtype="int32") 
    x = NucleotideEmbedding(dim)(tokens)  
    x = layers.Conv1D(dim, kernel_size=3, padding="same", activation="relu")(x)
    x = layers.Dense(n_outputs)(x)  
    if n_outputs == 1:
        x = layers.Lambda(lambda t: ops.squeeze(t, axis=-1))(x) 
    return keras.Model(tokens, x, name="mini_model")
